### Purpose

1. API Gateway(REST)에 연결된 Lambda 함수 목록을 찾는다
2. 각 Lambda의 소스 코드(zip)를 다운로드해서 `Downloaded/{날짜}/{함수명}/` 에 풀어 둔다
3. 함수 설정(runtime, handler, timeout, 환경변수 **키 이름만**)을 `_config.json` 으로 저장한다

- 환경변수 **값은 저장하지 않는다** (인증키가 섞여 있을 수 있음)
- 읽기 전용 호출만 사용 (`apigateway:GET`, `lambda:GetFunction`)

In [ ]:
import os
import re
import io
import json
import zipfile
import urllib.request
from datetime import datetime

import pandas as pd

# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 (OS 인증서 저장소 사용)
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    print("truststore 없음 - SSL 오류가 나면: pip install truststore")

import boto3
boto3.setup_default_session(profile_name='default', region_name='us-east-1')

Settings

In [ ]:
# 대상 API Gateway (docs/DATA_AND_API.md 참고)
API_IDS = {
    'jyipsj28s9': '컴포넌트 관리 (약품/장비/인력/유지보수)',
    'spcxatxbph': '기본 설정 및 작업 기록',
}

# API Gateway에 안 걸린 함수도 받고 싶으면 여기에 이름 추가
EXTRA_FUNCTIONS = []

# None 이면 전체, 일부만 받으려면 예: ['dschemical']  (부분 문자열 매칭)
ONLY = None

AWS_folder = "./"
today_date = datetime.now().strftime('%Y%m%d')
out_folder = os.path.join(AWS_folder, 'Downloaded', f'lambda{today_date}')
os.makedirs(out_folder, exist_ok=True)
out_folder

API Gateway → Lambda 매핑 찾기

In [ ]:
apigw = boto3.client('apigateway')

def list_api_lambdas(api_id):
    rows = []
    api_name = apigw.get_rest_api(restApiId=api_id)['name']
    paginator = apigw.get_paginator('get_resources')
    for page in paginator.paginate(restApiId=api_id, embed=['methods']):
        for res in page['items']:
            for method, m in (res.get('resourceMethods') or {}).items():
                uri = (m.get('methodIntegration') or {}).get('uri', '')
                found = re.search(r'function:([^/:]+)', uri)
                if found:
                    rows.append({'api_id': api_id, 'api_name': api_name, 'path': res['path'],
                                 'method': method, 'function': found.group(1)})
    return rows

rows = []
for api_id in API_IDS:
    rows += list_api_lambdas(api_id)

df_map = pd.DataFrame(rows).sort_values(['api_id', 'path', 'method']).reset_index(drop=True)
df_map.to_csv(os.path.join(out_folder, '_api_lambda_map.csv'), index=False, encoding='utf-8-sig')
df_map

In [ ]:
# 함수별로 묶어서 보기
functions = sorted(set(df_map['function']) | set(EXTRA_FUNCTIONS))
if ONLY:
    functions = [f for f in functions if any(k.lower() in f.lower() for k in ONLY)]

df_fn = (df_map.groupby('function')
         .agg(api=('api_id', 'first'), paths=('path', lambda s: ', '.join(sorted(set(s)))))
         .reindex(functions))
df_fn

Lambda 소스 다운로드

In [ ]:
lam = boto3.client('lambda')

CONFIG_KEYS = ['FunctionName', 'Runtime', 'Handler', 'Timeout', 'MemorySize', 'Role',
               'LastModified', 'CodeSha256', 'CodeSize', 'Architectures', 'PackageType']

def download_lambda(fn_name, dest_root):
    info = lam.get_function(FunctionName=fn_name)
    conf = info['Configuration']
    dest = os.path.join(dest_root, fn_name)
    os.makedirs(dest, exist_ok=True)

    # 설정 저장 - 환경변수는 키 이름만
    saved = {k: conf.get(k) for k in CONFIG_KEYS if k in conf}
    saved['EnvironmentKeys'] = sorted((conf.get('Environment') or {}).get('Variables', {}).keys())
    saved['Layers'] = [l['Arn'] for l in conf.get('Layers', [])]
    saved['VpcSubnets'] = (conf.get('VpcConfig') or {}).get('SubnetIds', [])
    with open(os.path.join(dest, '_config.json'), 'w', encoding='utf-8') as f:
        json.dump(saved, f, ensure_ascii=False, indent=2)

    if conf.get('PackageType') == 'Image':
        return {'function': fn_name, 'files': 0, 'note': '컨테이너 이미지 - 코드 다운로드 불가'}

    # 코드 zip (presigned URL, 10분 유효)
    with urllib.request.urlopen(info['Code']['Location']) as r:
        data = r.read()
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall(dest)
        n_files = len(z.namelist())

    return {'function': fn_name, 'runtime': conf.get('Runtime'), 'handler': conf.get('Handler'),
            'last_modified': conf.get('LastModified', '')[:19], 'files': n_files,
            'size_kb': round(len(data) / 1024, 1), 'note': ''}

results = []
for fn in functions:
    try:
        results.append(download_lambda(fn, out_folder))
        print('OK  ', fn)
    except Exception as e:
        results.append({'function': fn, 'note': f'실패: {e}'})
        print('FAIL', fn, e)

df_result = pd.DataFrame(results)
df_result

다운로드된 파일 확인

- 라이브러리를 같이 묶어 올린 함수는 파일 수가 많다. 직접 작성한 코드는 보통 `lambda_function.py`

In [ ]:
for fn in functions:
    d = os.path.join(out_folder, fn)
    if not os.path.isdir(d):
        continue
    top = sorted(f for f in os.listdir(d) if f != '_config.json')
    print(f"{fn:30} {top[:6]}{' ...' if len(top) > 6 else ''}")